# Pandas Session 3 Assignment: Superstore Sales Analysis

## GroupBy, Aggregation, Time Series and Pivot Tables

**Dataset:** Sample Superstore Sales

**Source:** Kaggle - https://www.kaggle.com/datasets/vivek468/superstore-dataset-final

---

### Learning Objectives

This assignment tests your understanding of Session 3 concepts:

- GroupBy operations (split-apply-combine)
- Aggregation with agg() and named aggregations
- Transform vs aggregation
- Pivot tables and crosstab
- Time series operations (to_datetime, resample, rolling)
- Shift for lag features

---

### Dataset Description

The Superstore dataset contains sales transactions from a retail store.

**Key Columns:**

- Order ID, Order Date, Ship Date - Transaction info
- Customer ID, Customer Name, Segment - Customer info
- Country, City, State, Region - Geography
- Product ID, Category, Sub-Category, Product Name - Product info
- Sales, Quantity, Discount, Profit - Metrics

---

### Business Context

You are a data analyst at Superstore. Management wants answers to:

- Which regions/categories are most profitable?
- What are the sales trends over time?
- Which customer segments perform best?
- How can we identify top and bottom performers?

**Total Points: 100 (+ 10 bonus)**

---
## Part 1: Data Loading and Preparation (10 points)
---

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("Sample - Superstore.csv", encoding="latin-1")

print("Dataset loaded!")
print(f"Shape: {df.shape}")

df.head()

### Task 1.1: Data Preparation (5 points)

1. Convert 'Order Date' and 'Ship Date' to datetime
2. Create 'Order_Year', 'Order_Month', 'Order_Quarter' columns from Order Date
3. Set 'Order Date' as the index (keep a copy of the original df)

In [ ]:
df["Order Date"] = pd.to_datetime(df["Order Date"])

df["Ship Date"] = pd.to_datetime(df["Ship Date"])

df["Order_Year"] = df["Order Date"].dt.year

df["Order_Month"] = df["Order Date"].dt.month

df["Order_Quarter"] = df["Order Date"].dt.quarter

df_original = df.copy()

df = df.set_index("Order Date")

df.head()

### Task 1.2: Initial Exploration (5 points)

1. How many unique customers, products, and orders?
2. What is the date range of the data?
3. What are the unique values in Category and Region?

In [ ]:
df["Customer ID"].nunique()

df["Product ID"].nunique()

df["Order ID"].nunique()

df.index.min(), df.index.max()

df["Category"].unique()

df["Region"].unique()

---
## Part 2: Basic GroupBy Operations (20 points)
---

### Task 2.1: Single Column GroupBy (5 points)

1. Calculate total Sales by Region
2. Calculate average Profit by Category
3. Count number of orders by Segment

In [ ]:
df.groupby("Region")["Sales"].sum()

df.groupby("Category")["Profit"].mean()

df.groupby("Segment")["Order ID"].count()

### Task 2.2: Multiple Column GroupBy (5 points)

1. Calculate total Sales by Region AND Category
2. Calculate average Discount by Category AND Sub-Category
3. Find the count of orders by Year AND Quarter

In [ ]:
df.groupby(["Region", "Category"])["Sales"].sum()

df.groupby(["Category", "Sub-Category"])["Discount"].mean()

df.groupby(["Order_Year", "Order_Quarter"])["Order ID"].count()

### Task 2.3: GroupBy with Multiple Aggregations (5 points)

For each Category, calculate:

- Total Sales (sum)
- Average Profit (mean)
- Number of transactions (count)
- Maximum single sale (max)

In [ ]:
df.groupby("Category").agg({
    "Sales": "sum",
    "Profit": "mean",
    "Order ID": "count",
    "Sales": "max"
})

### Task 2.4: Named Aggregations (5 points)

Use named aggregations to create a clean summary by Region:

- total_sales: sum of Sales
- avg_profit: mean of Profit
- total_quantity: sum of Quantity
- order_count: count of Order ID
- avg_discount: mean of Discount

In [ ]:
df.groupby("Region").agg(
    total_sales=("Sales", "sum"),
    avg_profit=("Profit", "mean"),
    total_quantity=("Quantity", "sum"),
    order_count=("Order ID", "count"),
    avg_discount=("Discount", "mean")
)

---
## Part 3: Advanced Aggregation (15 points)
---

### Task 3.1: Custom Aggregation Functions (5 points)

1. Calculate the profit margin (Profit/Sales * 100) for each Category
2. Find the range (max - min) of Sales for each Region
3. Calculate the coefficient of variation (std/mean) of Profit by Segment

In [ ]:
profit_margin = df.groupby("Category").apply(
    lambda x: (x["Profit"].sum() / x["Sales"].sum()) * 100
)

sales_range = df.groupby("Region")["Sales"].agg(
    lambda x: x.max() - x.min()
)

profit_cv = df.groupby("Segment")["Profit"].agg(
    lambda x: x.std() / x.mean()
)

profit_margin

sales_range

profit_cv

### Task 3.2: Top N Analysis (5 points)

1. Find the top 5 customers by total Sales
2. Find the top 3 products by total Profit in each Category
3. Find the bottom 5 Sub-Categories by average Profit

In [ ]:
top_5_customers = df.groupby("Customer Name")["Sales"].sum().nlargest(5)

top_3_products = (
    df.groupby(["Category", "Product Name"])["Profit"]
    .sum()
    .groupby(level=0)
    .nlargest(3)
)

bottom_5_subcategories = (
    df.groupby("Sub-Category")["Profit"]
    .mean()
    .nsmallest(5)
)

top_5_customers

top_3_products

bottom_5_subcategories

### Task 3.3: Filter Groups (5 points)

1. Filter to only Categories with total Sales > 500,000
2. Filter to only States with more than 100 orders
3. Find Customers who have made purchases in all 4 Regions

In [ ]:
high_sales_categories = (
    df.groupby("Category")["Sales"]
    .sum()
    .loc[lambda x: x > 500000]
)

states_more_100_orders = (
    df.groupby("State")["Order ID"]
    .count()
    .loc[lambda x: x > 100]
)

customers_all_regions = (
    df.groupby("Customer Name")["Region"]
    .nunique()
    .loc[lambda x: x == 4]
)

high_sales_categories

states_more_100_orders

customers_all_regions

---
## Part 4: Transform Operations (15 points)
---

### Task 4.1: Basic Transform (5 points)

1. Add a column 'Category_Avg_Sales' showing the average sales for that row's category
2. Add a column 'Region_Total_Profit' showing total profit for that row's region
3. Verify that transform returns same-length output as input

In [ ]:
df["Category_Avg_Sales"] = df.groupby("Category")["Sales"].transform("mean")

df["Region_Total_Profit"] = df.groupby("Region")["Profit"].transform("sum")

print(len(df["Category_Avg_Sales"]) == len(df))
print(len(df["Region_Total_Profit"]) == len(df))

df[["Category", "Sales", "Category_Avg_Sales",
    "Region", "Profit", "Region_Total_Profit"]].head()

### Task 4.2: Standardization within Groups (5 points)

1. Create 'Sales_Zscore_by_Category': standardize Sales within each Category
   Formula: (value - group_mean) / group_std
2. Create 'Profit_Pct_of_Region': each row's profit as percentage of its region's total
3. Rank products by Sales within each Sub-Category

In [ ]:
df["Sales_Zscore_by_Category"] = df.groupby("Category")["Sales"].transform(
    lambda x: (x - x.mean()) / x.std()
)

df["Profit_Pct_of_Region"] = (
    df["Profit"] / df.groupby("Region")["Profit"].transform("sum")
) * 100

df["Sales_Rank_in_SubCategory"] = df.groupby("Sub-Category")["Sales"].rank(
    ascending=False,
    method="dense"
)

df[[
    "Category",
    "Sales",
    "Sales_Zscore_by_Category",
    "Region",
    "Profit",
    "Profit_Pct_of_Region",
    "Sub-Category",
    "Sales_Rank_in_SubCategory"
]].head()

### Task 4.3: Cumulative Operations (5 points)

1. Calculate cumulative Sales by Customer (running total per customer)
2. Calculate the rank of each order by Sales within each Customer
3. Calculate the cumulative count of orders per Region

In [ ]:
df["Cumulative_Sales_by_Customer"] = df.groupby("Customer Name")["Sales"].cumsum()

df["Sales_Rank_by_Customer"] = df.groupby("Customer Name")["Sales"].rank(
    ascending=False,
    method="dense"
)

df["Cumulative_Orders_by_Region"] = df.groupby("Region").cumcount() + 1

df[[
    "Customer Name",
    "Sales",
    "Cumulative_Sales_by_Customer",
    "Sales_Rank_by_Customer",
    "Region",
    "Cumulative_Orders_by_Region"
]].head()

---
## Part 5: Pivot Tables (15 points)
---

### Task 5.1: Basic Pivot Tables (5 points)

1. Create pivot table: Sales by Region (rows) and Category (columns)
2. Create pivot table: Average Profit by Segment (rows) and Year (columns)
3. Add margins (totals) to the first pivot table

In [ ]:
pivot_sales = pd.pivot_table(
    df,
    values="Sales",
    index="Region",
    columns="Category",
    aggfunc="sum"
)

pivot_profit = pd.pivot_table(
    df,
    values="Profit",
    index="Segment",
    columns="Order_Year",
    aggfunc="mean"
)

pivot_sales_total = pd.pivot_table(
    df,
    values="Sales",
    index="Region",
    columns="Category",
    aggfunc="sum",
    margins=True
)

print(pivot_sales)

print(pivot_profit)

print(pivot_sales_total)

### Task 5.2: Multi-Aggregation Pivot Tables (5 points)

Create a pivot table showing:

- Rows: Category
- Columns: Region
- Values: Both sum and mean of Sales

In [ ]:
pivot_multi = pd.pivot_table(
    df,
    values="Sales",
    index="Category",
    columns="Region",
    aggfunc=["sum", "mean"]
)

print(pivot_multi)

### Task 5.3: Crosstab Analysis (5 points)

1. Create a crosstab of Region vs Category (counts)
2. Create a crosstab of Segment vs Ship Mode with normalized values (by row)
3. What percentage of Corporate customers use Standard Class shipping?

In [ ]:
df_temp = df.reset_index()

region_category = pd.crosstab(
    df_temp["Region"],
    df_temp["Category"]
)

segment_shipmode = pd.crosstab(
    df_temp["Segment"],
    df_temp["Ship Mode"],
    normalize="index"
)

corporate_standard_percentage = (
    segment_shipmode.loc["Corporate", "Standard Class"] * 100
)

print(region_category)

print(segment_shipmode)

print("Percentage of Corporate customers using Standard Class:")
print(corporate_standard_percentage)

---
## Part 6: Time Series Analysis (20 points)
---

### Task 6.1: Time-Based Selection (5 points)

Using the datetime index:

1. Select all orders from 2017
2. Select orders from Q4 of any year
3. Select orders between March and June 2016

In [ ]:
orders_2017 = df.loc["2017"]

print(orders_2017.head())


orders_q4 = df[df.index.quarter == 4]

print(orders_q4.head())


orders_mar_jun_2016 = df[
    (df.index >= "2016-03-01") &
    (df.index <= "2016-06-30")
]

print(orders_mar_jun_2016.head())

### Task 6.2: Resampling (5 points)

1. Resample Sales to monthly totals
2. Resample Profit to quarterly averages
3. Find the month with highest total Sales

In [ ]:
df = df.sort_index()

monthly_sales = df["Sales"].resample("ME").sum()

print(monthly_sales.head())

quarterly_profit = df["Profit"].resample("QE").mean()

print(quarterly_profit.head())

highest_sales_month = monthly_sales.idxmax()
highest_sales_value = monthly_sales.max()

print(highest_sales_month)
print(highest_sales_value)

### Task 6.3: Rolling Windows (5 points)

1. Calculate 7-day rolling average of Sales
2. Calculate 30-day rolling sum of Quantity
3. Calculate 90-day rolling standard deviation of Profit

In [ ]:
sales_7day_avg = df["Sales"].rolling("7D").mean()

print(sales_7day_avg.head())

quantity_30day_sum = df["Quantity"].rolling("30D").sum()

print(quantity_30day_sum.head())

profit_90day_std = df["Profit"].rolling("90D").std()

print(profit_90day_std.head())

### Task 6.4: Shift and Lag Features (5 points)

1. Create a column showing previous month's total Sales (lag 1 month)
2. Create a column showing Sales change from previous month
3. Create a column showing Sales percentage change from previous month

In [ ]:
monthly_sales = df["Sales"].resample("ME").sum()

previous_month_sales = monthly_sales.shift(1)

sales_change = monthly_sales - previous_month_sales

sales_percentage_change = monthly_sales.pct_change() * 100

print(previous_month_sales.head())

print(sales_change.head())

print(sales_percentage_change.head())

---
## Part 7: Business Analysis Questions (5 points)
---

Answer these business questions with code:

1. Which Region-Category combination has the highest profit margin?
2. Is there seasonality in sales? Which quarter performs best?
3. What is the year-over-year growth rate for each Category?
4. Which customers have increasing purchase trends?
5. What percentage of total profit comes from each Segment?

In [ ]:
profit_margin = (
    df.groupby(["Region", "Category"])
    .apply(lambda x: (x["Profit"].sum() / x["Sales"].sum()) * 100)
)

print(profit_margin.sort_values(ascending=False).head(1))


quarter_sales = df.groupby("Order_Quarter")["Sales"].sum()

print(quarter_sales)
print(quarter_sales.idxmax())


year_category_sales = (
    df.groupby(["Order_Year", "Category"])["Sales"]
    .sum()
    .unstack()
)

yoy_growth = year_category_sales.pct_change() * 100

print(yoy_growth)


customer_year_sales = (
    df.groupby(["Customer Name", "Order_Year"])["Sales"]
    .sum()
    .unstack()
)

increasing_customers = customer_year_sales[
    customer_year_sales.diff(axis=1).iloc[:, 1:].gt(0).all(axis=1)
]

print(increasing_customers.index)


segment_profit_percentage = (
    df.groupby("Segment")["Profit"].sum()
    / df["Profit"].sum()
    * 100
)

print(segment_profit_percentage)

### Your Findings:

1. Best Region-Category:
2. Best Quarter:
3. YoY Growth:
4. Growing Customers:
5. Profit by Segment:

---
## Bonus: Executive Dashboard Data (10 points)
---

Create a comprehensive summary DataFrame that could power an executive dashboard:

1. Monthly KPIs: Sales, Profit, Orders, Avg Order Value
2. Include MoM (month-over-month) change percentages
3. Include YTD (year-to-date) cumulative totals
4. Include 3-month rolling averages

In [ ]:
dashboard = pd.DataFrame()

dashboard["Sales"] = df["Sales"].resample("ME").sum()
dashboard["Profit"] = df["Profit"].resample("ME").sum()
dashboard["Orders"] = df["Order ID"].resample("ME").count()

dashboard["Avg_Order_Value"] = (
    dashboard["Sales"] / dashboard["Orders"]
)

dashboard["Sales_MoM_%"] = dashboard["Sales"].pct_change() * 100
dashboard["Profit_MoM_%"] = dashboard["Profit"].pct_change() * 100
dashboard["Orders_MoM_%"] = dashboard["Orders"].pct_change() * 100

dashboard["Sales_YTD"] = dashboard["Sales"].groupby(dashboard.index.year).cumsum()
dashboard["Profit_YTD"] = dashboard["Profit"].groupby(dashboard.index.year).cumsum()
dashboard["Orders_YTD"] = dashboard["Orders"].groupby(dashboard.index.year).cumsum()

dashboard["Sales_3M_Rolling_Avg"] = dashboard["Sales"].rolling(3).mean()
dashboard["Profit_3M_Rolling_Avg"] = dashboard["Profit"].rolling(3).mean()
dashboard["Orders_3M_Rolling_Avg"] = dashboard["Orders"].rolling(3).mean()

print(dashboard.head())
print(dashboard.shape)

---
## Submission Checklist

- [ ] All groupby operations completed correctly
- [ ] Transform vs agg distinction demonstrated
- [ ] Pivot tables created with proper structure
- [ ] Time series operations (resample, rolling, shift) working
- [ ] Business questions answered with insights
- [ ] Code is clean and commented

**Total Points: 100 (+ 10 bonus)**